# Geracao dos artefatos da camada Gold

Este notebook recompõe os artefatos atuais do contrato setorial de `Agricultura (Acucar, Alcool e Cana)` dentro de `notebooks/03_gold/`.

Arquivos gerados:
- `seed_contrato_variaveis_agricultura_acucar_alcool_cana.csv`
- `seed_contrato_variaveis_agricultura_acucar_alcool_cana.md`


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks' / '03_gold').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do repositorio.')


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'notebooks' / '03_gold'
CSV_PATH = OUTPUT_DIR / 'seed_contrato_variaveis_agricultura_acucar_alcool_cana.csv'
MD_PATH = OUTPUT_DIR / 'seed_contrato_variaveis_agricultura_acucar_alcool_cana.md'

CSV_COLUMNS = [
    'codigo_da_variavel',
    'variavel',
    'origem',
    'cd_conta',
    'ds_conta',
    'tabela_silver',
    'st_conta_fixa',
    'cobertura_no_setor',
    'coalesce',
    'empresas_sem_variavel',
    'impacto_quando_ausente',
    'validacao_setor_piloto',
    'usado_em',
    'atencao',
]

VARIAVEL_TITULOS = {
    'V01': 'Ativo Total',
    'V02': 'Ativo Circulante',
    'V03': 'Ativo Nao Circulante',
    'V04': 'Realizavel a Longo Prazo',
    'V05': 'Imobilizado',
    'V06': 'Estoques',
    'V07': 'Contas a Receber',
    'V08': 'Fornecedores',
    'V09': 'Passivo Circulante',
    'V10': 'Passivo Nao Circulante',
    'V11': 'Patrimonio Liquido',
    'V12': 'Emprestimos e Financiamentos CP',
    'V13': 'Emprestimos e Financiamentos LP',
    'V14': 'Caixa e Equivalentes BP',
    'V15': 'Aplicacoes Financeiras',
    'V16': 'Receita Liquida de Vendas',
    'V17': 'CPV CMV COGS',
    'V18': 'Lucro Bruto',
    'V19': 'EBIT Resultado Operacional',
    'V20': 'Lucro Liquido do Exercicio',
    'V21': 'LPA Basico ON ou Diluido ON',
    'V22': 'Depreciacao e Amortizacao',
    'V23': 'Caixa Liquido das Atividades Operacionais',
    'V24': 'Saldo Final de Caixa DFC',
    'V25': 'EBITDA',
    'V26': 'Ativo Circulante Financeiro Fleuriet',
    'V27': 'Divida Bruta',
    'V28': 'Divida Liquida',
    'V29': 'Capital Investido',
}

NOTA_N1 = [
    ('Liquidez Imediata', '`1.01.01` (BP)', '`6.05.02` (DFC)', 'O ratio e definido sobre o balanco patrimonial'),
    ('Divida Liquida', '`6.05.02` (DFC)', '`1.01.01` (BP)', 'A DFC incorpora equivalentes de caixa e efeitos de `CPC 03` e `CPC 31`'),
    ('Ativo Circulante Financeiro Fleuriet', '`1.01.01 + 1.01.02`', '`6.05.02` isolado', 'O modelo Fleuriet parte do balanco e nao do fluxo'),
]

MAPA_RAPIDO = [
    ('Liquidez Geral', '`(AC + RLP) / (PC + ELP)`', '`V02`, `V04`, `V09`, `V10`'),
    ('Liquidez Corrente', '`AC / PC`', '`V02`, `V09`'),
    ('Liquidez Seca', '`(AC - Estoques) / PC`', '`V02`, `V06`, `V09`'),
    ('Liquidez Imediata', '`Disponibilidades / PC`', '`V14`, `V09`'),
    ('PCT/CP', '`(PC + ELP) / PL`', '`V09`, `V10`, `V11`'),
    ('PCT/AT', '`(PC + ELP) / AT`', '`V09`, `V10`, `V01`'),
    ('Garantia CP/CT', '`PL / (PC + ELP)`', '`V11`, `V09`, `V10`'),
    ('Composicao de Endividamento', '`PC / (PC + ELP)`', '`V09`, `V10`'),
    ('Imobilizacao do CP', '`Imobilizado / PL`', '`V05`, `V11`'),
    ('Imobilizacao do AT', '`Imobilizado / AT`', '`V05`, `V01`'),
    ('Margem Bruta', '`Lucro Bruto / Receita Liquida`', '`V18`, `V16`'),
    ('Margem Operacional', '`EBIT / Receita Liquida`', '`V19`, `V16`'),
    ('Margem Liquida', '`Lucro Liquido / Receita Liquida`', '`V20`, `V16`'),
    ('LPA', '`Lucro Liquido / Numero de acoes`', '`V21` com suporte de conciliacao de `V20`'),
    ('ROA', '`Lucro Liquido / Ativo Total`', '`V20`, `V01`'),
    ('ROE', '`Lucro Liquido / PL`', '`V20`, `V11`'),
    ('ROI', '`Lucro Liquido / Capital Investido`', '`V20`, `V29`'),
    ('Giro dos Estoques', '`CPV / Estoques`', '`V17`, `V06`'),
    ('Giro de Contas a Receber', '`Receita Liquida / Contas a Receber`', '`V16`, `V07`'),
    ('Giro de Contas a Pagar', '`CPV / Fornecedores`', '`V17`, `V08`'),
    ('Giro do Ativo Circulante', '`Receita Liquida / AC`', '`V16`, `V02`'),
    ('PMRE', '`(Estoques * 360) / CPV`', '`V06`, `V17`'),
    ('PMRV', '`(Contas a Receber * 360) / Receita Liquida`', '`V07`, `V16`'),
    ('PMPC', '`(Fornecedores * 360) / CPV`', '`V08`, `V17`'),
    ('PMRAC', '`(AC * 360) / Receita Liquida`', '`V02`, `V16`'),
    ('Ciclo Economico', '`PMRE + PMRV`', '`V06`, `V07`, `V16`, `V17`'),
    ('Ciclo Financeiro', '`PMRE + PMRV - PMPC`', '`V06`, `V07`, `V08`, `V16`, `V17`'),
    ('CGL CCL', '`AC - PC`', '`V02`, `V09`'),
    ('NCG', '`(AC - Caixa - Aplicacoes) - (PC - Emprestimos CP)`', '`V02`, `V14`, `V15`, `V09`, `V12`'),
    ('Saldo de Tesouraria', '`Ativo Circulante Financeiro - Passivo Circulante Financeiro`', '`V26`, `V12`'),
]

PENDENCIAS_ANOMALIAS = [
    'empresas sem estoque',
    'empresas com variantes problematicas em `3.99.*`',
    'empresas com `PL` negativo',
    'empresas com divergencias relevantes entre `1.01.01` e `6.05.02`',
]

CHECKLIST = [
    'Todas as variaveis do bloco do Excel foram convertidas em fichas `V01` a `V29`.',
    'As notas `N1` a `N4` foram explicitadas.',
    'O `Mapa Rapido` cobre os 7 grupos de indicadores.',
    'Ainda falta a etapa de validacao real na `silver` do setor para preencher cobertura, empresas-anomalia e validacao contra empresa piloto.',
]


def read_rows(csv_path: Path) -> list[dict[str, str]]:
    if not csv_path.exists():
        raise FileNotFoundError(f'Arquivo seed nao encontrado: {csv_path}')
    with csv_path.open('r', encoding='utf-8', newline='') as fp:
        rows = list(csv.DictReader(fp))
    expected = set(CSV_COLUMNS)
    received = set(rows[0].keys()) if rows else set()
    if expected != received:
        raise ValueError(f'Colunas inesperadas no CSV. Esperado={sorted(expected)} Recebido={sorted(received)}')
    return rows


def escape_md(value: str) -> str:
    return str(value).replace('|', '\\|')


def render_table(headers: list[str], rows: list[tuple[str, ...]]) -> list[str]:
    rendered = [
        '| ' + ' | '.join(headers) + ' |',
        '|'+ '|'.join(['---'] * len(headers)) + '|',
    ]
    for row in rows:
        rendered.append('| ' + ' | '.join(escape_md(cell) for cell in row) + ' |')
    return rendered


def field_rows(row: dict[str, str]) -> list[tuple[str, str]]:
    return [
        ('CD_CONTA', row['cd_conta']),
        ('DS_CONTA', row['ds_conta']),
        ('Tabela Silver', row['tabela_silver']),
        ('ST_CONTA_FIXA', row['st_conta_fixa']),
        ('Cobertura no setor', row['cobertura_no_setor']),
        ('COALESCE?', row['coalesce']),
        ('Empresas sem a variavel', row['empresas_sem_variavel']),
        ('Impacto quando ausente', row['impacto_quando_ausente']),
        ('Validacao no setor piloto', row['validacao_setor_piloto']),
        ('Usado em', row['usado_em']),
        ('Atencao', row['atencao']),
    ]


def render_markdown(rows: list[dict[str, str]]) -> str:
    lines: list[str] = []
    lines.append('# Premissas Setor Agricultura Acucar Alcool e Cana')
    lines.append('')
    lines.append('## Cabecalho')
    lines.append('')
    lines.extend(
        render_table(
            ['Campo', 'Valor'],
            [
                ('Setor', 'Agricultura (Acucar, Alcool e Cana)'),
                ('Squad', 'A definir'),
                ('Data', '2026-05-21'),
                ('Base de dados', '`layer_02_silver.n1_dfp_cia_aberta_bp`, `layer_02_silver.n1_dfp_cia_aberta_dre`, `layer_02_silver.n1_dfp_cia_aberta_dfc`'),
                ('Escopo', 'Mapeamento inicial das variaveis necessarias para calcular os indicadores classicos do material `indicadores_financeiros_1.html`'),
                ('Status', 'Rascunho estrutural aderente ao formato do HTML; ainda depende de validacao setorial direta na `silver`'),
            ],
        )
    )
    lines.append('')
    lines.append('## Notas Gerais Obrigatorias')
    lines.append('')
    lines.append('### N1. Caixa para indicadores: usar BP ou DFC?')
    lines.append('')
    lines.extend(render_table(['Indicador', 'Conta correta', 'Conta incorreta', 'Motivo'], NOTA_N1))
    lines.append('')
    lines.append('### N2. Estoques ausentes')
    lines.append('')
    lines.append('Cobertura no setor: `A validar na silver do setor`.')
    lines.append('')
    lines.append('Regra:')
    lines.append('- Usar `COALESCE(estoques, 0)` apenas em `Liquidez Seca`.')
    lines.append('- Usar `N/A` para `Giro de Estoques`, `PMRE`, `Ciclo Economico` e `Ciclo Financeiro` quando nao houver estoque reportado.')
    lines.append('- Registrar nominalmente as empresas sem estoque na etapa de validacao setorial.')
    lines.append('')
    lines.append('### N3. LPA: usar leaves originais da familia 3.99')
    lines.append('')
    lines.append('Regra:')
    lines.append('- Usar somente contas leaf da familia `3.99.*`.')
    lines.append('- Nao usar pais reconstruidos de `3.99` como valor final de LPA.')
    lines.append('- Estrategia provisoria do contrato: priorizar `Diluido ON` quando existir; fallback para `Basico ON`; se a empresa nao reportar a classe ON, usar a leaf mais aderente e documentar a excecao.')
    lines.append('- Validar na `silver` do setor quantas variantes reais existem em `3.99.*`.')
    lines.append('')
    lines.append('### N4. ST_CONTA_FIXA: como interpretar')
    lines.append('')
    lines.append('Regra:')
    lines.append('- `S`: conta padronizada da taxonomia CVM; preferencia maxima no contrato.')
    lines.append('- `N`: conta livre da companhia; usar com cautela e sempre com validacao por `DS_CONTA_REPORTADA`.')
    lines.append('- `A validar`: familia candidata ainda nao confirmada no setor.')
    lines.append('- Variaveis derivadas ficam com `N/A`, porque nao nascem de uma unica conta bruta da CVM.')
    lines.append('')
    lines.append('## Fichas de Variaveis')
    lines.append('')
    for row in rows:
        codigo = row['codigo_da_variavel']
        titulo = VARIAVEL_TITULOS.get(codigo, row['variavel'].title())
        lines.append(f'### {codigo}. {titulo}')
        lines.append('')
        lines.extend(render_table(['Campo', 'Valor'], field_rows(row)))
        lines.append('')
    lines.append('## Mapa Rapido Indicador para Variaveis')
    lines.append('')
    lines.extend(render_table(['Indicador', 'Formula', 'Variaveis necessarias'], MAPA_RAPIDO))
    lines.append('')
    lines.append('## Empresas-anomalia identificadas')
    lines.append('')
    lines.append('Nenhuma empresa foi listada ainda nesta versao.')
    lines.append('')
    lines.append('Pendencias para preencher esta secao:')
    for item in PENDENCIAS_ANOMALIAS:
        lines.append(f'- {item}')
    lines.append('')
    lines.append('## Checklist de aderencia deste rascunho')
    lines.append('')
    for item in CHECKLIST:
        lines.append(f'- {item}')
    lines.append('')
    return '\n'.join(lines)


def write_csv(rows: list[dict[str, str]], output_path: Path) -> None:
    with output_path.open('w', encoding='utf-8', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=CSV_COLUMNS, quoting=csv.QUOTE_ALL, lineterminator='\n')
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row[column] for column in CSV_COLUMNS})


rows = read_rows(CSV_PATH)
if len(rows) != 29:
    raise ValueError(f'Esperadas 29 variaveis no seed atual, mas foram encontradas {len(rows)}.')

markdown_output = render_markdown(rows)
write_csv(rows, CSV_PATH)
MD_PATH.write_text(markdown_output, encoding='utf-8', newline='\n')

print(f'CSV gerado em: {CSV_PATH}')
print(f'Markdown gerado em: {MD_PATH}')
print(f'Total de variaveis: {len(rows)}')
